# 08. 문제 해결 — 이 구축에서 실제로 겪은 일들

증상 → 원인 → 확인 → 해결 순서입니다. 번호는 노트북 01~07 의 어디에서 겪는지 함께 적었습니다.

> 먼저 **한/영 키가 영어 상태인지**, `-o`/`-i`, `/`/`\`, `|` 오타를 의심하세요. 입력이 틀려 생긴 문제가 가장 많았습니다.

## 콘솔·입력
### 1. 서버 콘솔에서 한글이 `◆` 로 깨진다 (모든 단계)
- 원인: 서버에 모니터를 직접 연결한 화면(리눅스 콘솔)에는 한글 글꼴이 없음. SSH 나 PC 터미널에서는 정상.
- 해결: 스크립트 출력은 영어로 만들어 두었음. 한글 오류 문구(예: 계정 생성 거부 이유)는 규칙표로 하나씩 확인하거나 SSH 로 접속.

### 2. 명령이 `ㄴㅕㅇㅐ` 처럼 깨져 나온다
- 원인: 한/영 키가 **한글** 상태로 입력됨. 영어로 읽으면 원래 명령이 보입니다(`sudo` → `ㄴㅕㅇㅐ`).

### 3. `curl … -w '%{http_code}'` 가 `000` 과 이상한 출력을 낸다
- 원인: `-o`(영문 오)를 `-i`(아이)로 입력. curl 이 `/dev/null` 을 접속할 주소로 착각해 실패(`000`)하고 `-i` 때문에 헤더를 출력.
- 해결: `-o /dev/null`. 같은 이유로 `sed` 의 `/d` 를 `\d` 로 치면 `unterminated address regex`.

### 4. `read -r 변수` 를 쳤는데 커서만 깜빡이고 다음 명령이 안 된다
- 정상입니다. `read` 는 **입력을 기다리는** 명령이라 다음 줄에 값을 쳐야 합니다. `echo ${#변수}` 같은 다음 명령을 치면 그 글자가 값으로 들어갑니다 → `Ctrl+C`, `unset 변수` 로 정리하고 다시.
- 글자 수가 `0` 이면 값이 안 들어간 것(`read` 를 안 했거나 다른 탭에서 입력).

## 네트워크 (01~03)
### 5. 사설 IP 로 바꿨더니 이름 해석이 안 된다 (`getent hosts` 가 비어 있음, `ping 1.1.1.1` 은 됨)
- 원인: 통신사 공개 DNS 가 사설망에서 막힘. 확인: `resolvectl status [IFACE] | grep "DNS Servers"`.
- 해결: netplan 의 `nameservers` 에 **학교 DNS 를 먼저**, 공개 DNS(`1.1.1.1`, `8.8.8.8`)를 뒤에. `netplan try` 로 적용.
- `resolvectl dns` 로 넣은 임시 값은 재부팅하면 사라지니 netplan 에도 반영.

### 6. 안내문의 `10.x.x.254` 를 서버 IP 로 착각
- 안내문은 **PC 설정 화면 예시**였음. 서버 IP 는 학교가 서버마다 따로 부여한 번호. 확인: 사설 IP 를 임시로 붙여 게이트웨이에 `ping`(01 의 C).

### 7. 서버 자신이 자기 사이트 주소에서 `self-signed certificate`
- 원인과 우회: 06 의 C (`/etc/hosts` 임시 매핑).

## 설치·배포 (02, 04)
### 8. `bootstrap` 이 `ghcr.io` 에서 `error: 405` 로 멈춤
- 원인: 점검이 HTTP 오류 코드를 실패로 취급. 해결: 연결 여부만 보도록 수정됨(`curl -sS`, `-f` 없음).

### 9. ArgoCD 적용이 `metadata.annotations: Too long: may not be more than 262144 bytes`
- 해결: `kubectl apply --server-side --force-conflicts`.

### 10. ArgoCD 앱이 `Unknown`, `authorization failed: Write access to repository not granted`
- 원인: 서버에 넣은 토큰에 **`repo`** 권한이 없음(`read:packages` 만 있음).
- 해결: GitHub 의 토큰 편집 화면에서 `repo` 를 체크하고 **Update token**(값은 그대로). 이어서 `refresh=hard`.

### 11. 파드가 `ErrImagePull`/`ImagePullBackOff`, 이벤트에 `failed to fetch anonymous token … 401`
- 원인: `anonymous` = 로그인 정보 없이 접속. `ghcr-pull` 이 없거나 값이 비었거나, 토큰에 `read:packages` 가 없음.
- 확인: 토큰 글자 수 40, `curl -sI -H "Authorization: token $T1" https://api.github.com/user | grep -i "^x-oauth-scopes:"` 에 `read:packages, repo`.
- 해결: `ghcr-pull` 지우고 다시 만든 뒤 실패한 파드를 `delete pod -l app=…`. `crictl pull --creds …` 로 직접 시험하면 확실.

### 12. 앱이 `OutOfSync` 로 멈춰 새 변경이 안 올라온다
- 원인: 첫 동기화가 이미지 받기 실패 등으로 실패했고 그 기록이 남아 자동 동기화가 멈춤.
- 해결: `kubectl -n argocd delete application ssm` → `kubectl apply -f ~/ssm/deploy/argocd/application.yaml`(파드·DB 데이터 유지).

### 13. 처음 배포 때 마이그레이션 작업이 DB 가 없어 실패
- 원인: 마이그레이션을 배포 맨 처음 단계(PreSync)에 걸었음. 해결: 일반 동기화의 훅(Sync) + `sync-wave` 로 DB 가 준비된 **뒤**에 실행.
- 참고: 마이그레이션 작업은 끝나면 10분 뒤 자동 삭제되어 `get jobs` 가 비어 보임. 확인은 `alembic current`.

## 계정 (05)
### 14. `create-dept-head` 가 `command terminated with exit code 1`
- 원인: 입력 검증 실패(아이디 예약어 `dept-head`, 비밀번호 10자 미만·아이디 포함, 확인 불일치). 오류 문구가 한글이라 콘솔에서 안 보임.
- 해결: 05 의 규칙표를 하나씩 확인. 예약어가 아닌 아이디 사용.

### 15. 비밀번호를 잊었다
- 되돌릴 수 없는 해시라 볼 수 없음. 계정 삭제 후 재생성(05 의 A). 관리자 가입 폼에는 **비밀번호 확인 칸**이 있어 오타를 줄임.

## 서버 가입 (06)
### 16. 가입 마지막에 `ssh_ca: agent_error`, 웹 터미널이 "서버에 접속하지 못했습니다"
- 원인: 에이전트 서비스의 `ProtectSystem=full` 이 `/etc` 를 읽기 전용으로 만들어 `/etc/ssh/ssm_user_ca.pub`·계정 생성 실패(`Read-only file system`). `ReadWritePaths` 를 지정해도 이 서버에서는 효과가 없었음.
- 최신 스크립트: 서비스에서 `ProtectSystem` 을 제거. 이미 설치한 서버는 덮어쓰기 파일로 끄고 **재가입**:

In [ ]:
%%bash
sudo mkdir -p /etc/systemd/system/ssm-agent.service.d
sudo tee /etc/systemd/system/ssm-agent.service.d/rw.conf >/dev/null <<'EOF'
[Service]
ProtectSystem=no
EOF
sudo systemctl daemon-reload
sudo systemctl restart ssm-agent
systemctl show ssm-agent -p ProtectSystem        # ProtectSystem=no

### 17. `ssm-agent did not start` / `start-limit-hit`
- 원인: `ReadWritePaths` 에 적은 폴더(`/var/lib/ssm-agent`)가 **없으면** systemd 가 서비스를 시작하지 못함. 실패가 반복되면 `start-limit-hit` 로 시도 자체를 막음(결과일 뿐 원인이 아님).
- 확인: `sudo journalctl -u ssm-agent --no-pager | grep -iE "step|namespac|no such|exited" | head`
- 해결: 원인(폴더 생성) 수정 → `sudo systemctl reset-failed ssm-agent` → `restart`. 최신 스크립트는 폴더를 먼저 만들고 해당 옵션을 쓰지 않음.

### 18. 가입 토큰이 거절된다 (`403`, `could not download the agent package`)
- 원인: 43글자(예전) 토큰을 콘솔에서 직접 치다 틀림 / 30분 만료 / 재발급으로 이전 토큰 무효 / **이미 가입한 서버**에 새 토큰을 쓰려 함.
- 해결: 토큰이 **16글자 hex** 로 바뀜. 재발급은 한 번만. 토큰 검증용 `curl … agent.tar.gz?name=…&token=…` 으로 `200` 확인 후 실행.

### 19. 등록된 서버의 토큰을 다시 받을 수 없다
- 위쪽 "새 서버" 칸이 아니라 아래 **"등록된 서버" 표의 `재가입 명령`** 을 사용.

### 20. 가입 화면에서 `/etc/hosts` 임시 줄을 지우지 않았다
- 서버가 계속 인터넷 경로로 자기 사이트에 접속하게 되어 나중에 혼란. `grep tail /etc/hosts` 로 확인하고 `sed -i` 로 삭제.

## 사이트 사용 (07)
### 21. 재발급을 눌렀는데 새 값인지 모르겠다
- 사이트가 **"새 명령이 발급되었습니다 (시각, N번째)"** 알림과 큰 토큰(4글자씩 묶음)을 보여 줌. 키·토큰은 0 에 빗금이 있는 고정폭 글꼴로 표시(0 과 O 구분).

### 22. 사진·채팅에 비밀(토큰·키)이 노출됐다
- 즉시 교체: GitHub 토큰은 폐기하고 새로 발급, `SSM_SECRET_KEY` 는 아직 이를 쓰는 계정이 없을 때 새로 생성, 인증 앱 키는 계정을 지우고 재생성.